# FactIndic interactive summary and evidence check

Paste one **Roman-script Hinglish** article or passage. This live Colab demo generates an English summary, splits it into factual claims, and shows each claim beside source evidence.

## Run it

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells from the top. The first run downloads public model weights.
3. Paste Roman-script Hinglish, select a Qwen3 size and prompt, then click **Summarize and check claims**.
4. Follow the **Run progress** tracker beneath the controls. The green tick beside a Colab cell means the cell finished setting up the controls; model work begins only after you click the button and is tracked separately. If the claim checker returns malformed JSON, it retries once; if the retry also fails, the summary is preserved and marked unchecked.
5. Review the claim table and quoted evidence. Use the button to download a JSON report.

This is an **AI-assisted research demo**, not a factuality certificate. The selected Qwen3 model generates and checks its own summary. Evidence quotes are verified against the exact source text shown to the model, but quote presence does not prove logical support and the AI judgment can still be wrong. A claim with no quote or an unverifiable quote is labeled explicitly; number/date mismatches trigger one correction retry; if there is exactly one full date in both source and summary, a remaining date mismatch is replaced from the source and visibly disclosed. Other remaining numeric mismatches mark the summary as an unsafe draft. No factuality probability is invented. Human validation was skipped for this project version.

The fixed 30-source, 120-summary comparison remains in [`factindic_final_open_hinglish_colab.ipynb`](https://colab.research.google.com/github/Sirohi0/factindic/blob/main/factindic_final_open_hinglish_colab.ipynb). This separate notebook is the live single-input demonstration. Inputs and results stay in the Colab runtime unless you download the report; the notebook does not write them to Drive or the public repository.


## 1. Install the inference tools

Use a GPU runtime before running this cell. It downloads models for inference and does not train them.

In [ ]:
%pip -q install "transformers>=4.51.3,<5" accelerate sentencepiece ipywidgets


## 2. Initialize the GPU runtime

In [ ]:
import ast
import difflib
import gc
import hashlib
import html
import json
import re
import time
import unicodedata
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import torch
from IPython.display import clear_output, display, HTML, Markdown
from google.colab import files, output
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

output.enable_custom_widget_manager()
assert torch.cuda.is_available(), "Choose Runtime → Change runtime type → T4 GPU, then rerun this cell."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_IDS = {
    "Qwen3 1.7B": "Qwen/Qwen3-1.7B",
    "Qwen3 4B": "Qwen/Qwen3-4B",
}
NLI_MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
NLI_CONTRADICTION_THRESHOLD = 0.85
MAX_SOURCE_TOKENS = 2048
SUMMARY_TOKEN_LIMIT = 192
CLAIM_CHECK_TOKEN_LIMIT = 768
_loaded_model_key = None
_model = None
_tokenizer = None
_nli_model = None
_nli_tokenizer = None
_last_result = None

def load_model(model_key):
    global _loaded_model_key, _model, _tokenizer
    if _loaded_model_key == model_key and _model is not None:
        return _model, _tokenizer
    if _model is not None:
        del _model, _tokenizer
        _model, _tokenizer = None, None
        gc.collect()
        torch.cuda.empty_cache()
    model_id = MODEL_IDS[model_key]
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto"
    ).eval()
    # Qwen configs often carry sampling defaults. Set neutral values for greedy
    # decoding so Transformers does not emit misleading ignored-flag warnings.
    _model.generation_config.do_sample = False
    _model.generation_config.temperature = 1.0
    _model.generation_config.top_p = 1.0
    _model.generation_config.top_k = 50
    _loaded_model_key = model_key
    return _model, _tokenizer

def load_nli_model():
    global _nli_model, _nli_tokenizer
    if _nli_model is not None and _nli_tokenizer is not None:
        return _nli_model, _nli_tokenizer
    _nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_ID)
    _nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _nli_model = _nli_model.to(device).eval()
    return _nli_model, _nli_tokenizer

_HINGLISH_NLI_GLOSSES = (
    (r"\bbilkul na jaayein\b", "do not go at all"),
    (r"\bna jaayein\b", "do not go"),
    (r"\bdoor rehne ko kaha(?: gaya hai)?\b", "told to stay away"),
    (r"\bdoor rehne\b", "stay away"),
    (r"\bkhali kar dein\b", "evacuate immediately"),
    (r"\bband hai\b", "is closed"),
    (r"\balert jaari kiya hai\b", "issued an alert"),
    (r"\bkaafi suhana tha\b", "was quite pleasant"),
    (r"\bthandi hawa\b", "cool breeze"),
    (r"\bkoi bada jam nahi laga\b", "no major traffic jam occurred"),
    (r"\bnahi\b", "not"),
)

def normalize_hinglish_for_nli(text):
    """Map high-frequency Roman-script Hinglish cues to English glosses for XNLI scoring."""
    normalized = " ".join(str(text).split())
    for pattern, replacement in _HINGLISH_NLI_GLOSSES:
        normalized = re.sub(pattern, replacement, normalized, flags=re.I)
    return normalized

def detect_input_archetype(source_text, selected_mode="Auto-detect"):
    """Resolve whether the source is Devanagari/code-mixed or Roman-script Hinglish."""
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", str(source_text)))
    has_latin = bool(re.search(r"[A-Za-z]", str(source_text)))
    if selected_mode == "Devanagari / Code-Mixed (Hindi + English)":
        if not has_devanagari:
            raise ValueError("Selected Devanagari / Code-Mixed mode, but no Devanagari characters were found.")
        return "devanagari_codemixed"
    if selected_mode == "Roman-script Hinglish":
        if has_devanagari:
            raise ValueError("Selected Roman-script Hinglish mode, but Devanagari characters were found. Switch to Devanagari / Code-Mixed or Auto-detect.")
        if not has_latin:
            raise ValueError("No Latin-script text found. Paste a Roman-script Hinglish source.")
        return "roman_hinglish"
    if has_devanagari:
        return "devanagari_codemixed"
    if has_latin:
        return "roman_hinglish"
    raise ValueError("No Devanagari or Latin-script text found in the input.")

def nli_check_claim(premise, hypothesis, archetype="roman_hinglish"):
    if not premise or not hypothesis:
        return "unavailable", None
    model, tokenizer = load_nli_model()
    nli_premise = premise if archetype == "devanagari_codemixed" else normalize_hinglish_for_nli(premise)
    encoded = tokenizer(
        nli_premise, hypothesis, truncation=True, max_length=512, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        logits = model(**encoded).logits[0]
        probs = torch.softmax(logits, dim=-1)
        best_idx = int(torch.argmax(probs).item())
        best_prob = float(probs[best_idx].item())
    id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
    label = id2label.get(best_idx, "unknown")
    if archetype != "devanagari_codemixed" and label == "contradiction" and best_prob < NLI_CONTRADICTION_THRESHOLD:
        label = "uncertain_nli"
    return label, round(best_prob, 3)

def chat_generate(model, tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", enable_thinking=False
    ).to(model.device)
    input_tokens = int(encoded["input_ids"].shape[-1])
    started = time.perf_counter()
    with torch.inference_mode():
        # Use pure greedy decoding without n-gram blocking or repetition penalty so
        # entity names, dates, and exact source quotes can be copied verbatim.
        output_ids = model.generate(
            **encoded, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    elapsed = time.perf_counter() - started
    new_ids = output_ids[0][input_tokens:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip(), int(len(new_ids)), elapsed

def prepare_source(source, tokenizer):
    source_ids = tokenizer.encode(source, add_special_tokens=False)
    truncated = len(source_ids) > MAX_SOURCE_TOKENS
    source_used = tokenizer.decode(source_ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
    return source_used, len(source_ids), truncated

def extract_fact_anchors(source_text, archetype="roman_hinglish"):
    """Extract deterministic dates, numbers, and warning/negation cues before generation."""
    dates = [match.group(0) for match in _DATE_PATTERN.finditer(source_text)]
    numbers = numeric_tokens(source_text)
    cue_patterns = (
        r"bilkul na", r"na jaayein", r"door rehne", r"band hai",
        r"khali kar", r"\bmat\b", r"\bnahi\b", r"\bmana\b",
        r"\balert\b", r"\bwarning\b",
        r"नहीं", r"मत", r"बंद है", r"चेतावनी", r"अलर्ट", r"दूर रहें", r"खाली कर",
    )
    cues = []
    for pattern in cue_patterns:
        match = re.search(pattern, source_text, flags=re.I)
        if match:
            cues.append(match.group(0))
    return {"dates": dates, "numbers": numbers, "cues": cues, "archetype": archetype}

def make_summary(source_used, prompt_style, model, tokenizer, numeric_retry=False, fact_anchors=None, archetype="roman_hinglish"):
    source_desc = (
        "Devanagari Hindi / code-mixed Hindi-English"
        if archetype == "devanagari_codemixed"
        else "Roman-script Hinglish"
    )
    shared = (
        f"Translate and summarize the meaning of this {source_desc} source into a concise English "
        "summary of at most two sentences in natural English. Use past tense for completed events. "
        "Keep established names and technical terms exactly as spelled in the source; do not "
        "transliterate numbers into Roman numerals or invent suffixes. Keep the main event and the exact people, places, "
        "numbers, dates, attribution, and outcome stated in the source. Preserve negation, "
        "warnings, uncertainty, and directions exactly: do not reverse whether people should "
        "approach, avoid, leave, or remain in a place. Do not add advice, causes, identities, "
        "or other details. If a detail is unclear, leave it out. Return only the summary."
    )
    if prompt_style == "Factuality-aware":
        instruction = (
            shared + " Before answering, check that every clause is directly supported by "
            "the source and that no translation changed who did what or what people were told."
        )
    else:
        instruction = shared
    if numeric_retry:
        instruction += (
            " The previous draft changed a name, date, or number. For this retry, copy every "
            "digit sequence and date exactly as written in the source. Do not calculate, "
            "reformat, or guess a number or date; omit it if uncertain."
        )
    anchors = fact_anchors if fact_anchors is not None else extract_fact_anchors(source_used, archetype=archetype)
    anchor_lines = []
    if anchors.get("dates"):
        anchor_lines.append("Dates: " + ", ".join(anchors["dates"]))
    if anchors.get("numbers"):
        anchor_lines.append("Numbers: " + ", ".join(anchors["numbers"]))
    if anchors.get("cues"):
        anchor_lines.append("Warning/negation cues: " + ", ".join(anchors["cues"]))
    anchor_block = ("\nVerified source anchors (preserve exactly):\n" + "\n".join(anchor_lines)) if anchor_lines else ""
    prompt = f"{instruction}{anchor_block}\n\nSource text:\n{source_used}\n\nEnglish summary:"
    summary, tokens, seconds = chat_generate(model, tokenizer, prompt, SUMMARY_TOKEN_LIMIT)
    return summary, tokens, seconds, tokens >= SUMMARY_TOKEN_LIMIT

def normalize_quote(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).split()).casefold()

def parse_json_object(text):
    """Extract and parse one object; tolerate fences, prose and common loose JSON."""
    text = str(text).strip()
    start = text.find("{")
    if start < 0:
        raise ValueError("The model did not return a JSON object.")

    # Isolate the first balanced object, respecting both quote styles and escapes.
    stack = []
    quote = None
    escaped = False
    end = None
    pairs = {"}": "{", "]": "["}
    for index in range(start, len(text)):
        char = text[index]
        if quote:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == quote:
                quote = None
            continue
        if char in {"\"", "'"}:
            quote = char
        elif char in "{[":
            stack.append(char)
        elif char in "}]":
            if not stack or stack[-1] != pairs[char]:
                break
            stack.pop()
            if not stack:
                end = index + 1
                break
    if end is None:
        # Let the regular parsers produce a useful failure on truncated objects.
        candidate = text[start:]
    else:
        candidate = text[start:end]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Tolerate trailing commas that are invalid in strict JSON.
    cleaned = re.sub(r",\s*([}\]])", r"\1", candidate)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Some model responses use Python dict notation with single-quoted strings.
    try:
        return ast.literal_eval(cleaned)
    except (ValueError, SyntaxError):
        pass

    # Last common case: unquoted object keys. Quote only key-like tokens after
    # an opening brace or comma, then try strict JSON and Python literals again.
    quoted_keys = re.sub(r"([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1"\2":', cleaned)
    try:
        return json.loads(quoted_keys)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(quoted_keys)
        except (ValueError, SyntaxError) as error:
            raise ValueError("The model response is not a parseable claims object.") from error

def parse_claim_payload(text):
    parsed = parse_json_object(text)
    if not isinstance(parsed, dict) or not isinstance(parsed.get("claims"), list):
        raise ValueError("The model response is missing a JSON claims list.")
    return parsed

def check_claims(source_used, summary, model, tokenizer):
    prompt = (
        "You are checking whether a summary is supported by its source. Split it into up to "
        "12 short, atomic factual claims. For each claim choose exactly one status: supported "
        "(source explicitly supports it), contradicted (source explicitly says something "
        "incompatible), not_found (source does not state it), or unclear (source wording does "
        "not let you decide). For supported or contradicted claims, give one short evidence "
        "quote copied exactly from the source. For not_found or unclear claims, use an empty "
        "evidence string. If there are no factual claims to check, return an empty claims list. "
        "Never invent or rewrite a quote. Do not provide numerical probabilities or confidence "
        "scores. Return JSON only in this shape: "
        '{"claims":[{"claim":"...","status":"supported|contradicted|not_found|unclear",'
        '"evidence":"exact source quote or empty"}]}\n\n'
        f"SOURCE:\n{source_used}\n\nSUMMARY:\n{summary}\n"
    )
    raw, tokens, seconds = chat_generate(model, tokenizer, prompt, CLAIM_CHECK_TOKEN_LIMIT)
    try:
        parsed = parse_claim_payload(raw)
    except ValueError:
        # Retry once with a format-only correction. Keep the original source and
        # summary in context so the model cannot silently change what it checks.
        retry_prompt = (
            prompt + "\nFORMAT RETRY: Your previous response was invalid JSON. "
            "Return exactly one valid JSON object using double-quoted keys and strings, "
            "with no markdown, comments, trailing commas, or text before/after it. "
            "Keep the same factual judgments; do not invent or rewrite evidence.\n"
            "Previous response to correct:\n" + raw
        )
        retry_raw, retry_tokens, retry_seconds = chat_generate(
            model, tokenizer, retry_prompt, CLAIM_CHECK_TOKEN_LIMIT
        )
        tokens += retry_tokens
        seconds += retry_seconds
        raw = retry_raw
        try:
            parsed = parse_claim_payload(raw)
        except ValueError as error:
            raise ValueError(
                "The claim checker returned malformed JSON twice. The summary can still be "
                "shown, but no factuality conclusion is available."
            ) from error
    raw_claims = parsed["claims"]
    if not raw_claims:
        return [], tokens, seconds, raw
    allowed = {"supported", "contradicted", "not_found", "unclear"}
    normalized_source = normalize_quote(source_used)
    summary_numbers = set(numeric_tokens(summary))
    results = []
    for item in raw_claims[:12]:
        claim = str(item.get("claim", "")).strip()
        status = str(item.get("status", "unclear")).strip().lower()
        evidence = str(item.get("evidence", "")).strip()
        if not claim:
            continue
        if status not in allowed:
            status = "unclear"
        claim_numbers_not_in_summary = sorted(set(numeric_tokens(claim)) - summary_numbers)
        quote_found = bool(evidence) and normalize_quote(evidence) in normalized_source
        if claim_numbers_not_in_summary:
            # The checker added a number/date absent from the summary it was asked to split.
            # Do not treat any evidence judgment for this mismatched claim as trustworthy.
            status, evidence, quote_found = "checker_mismatch", "", False
        elif status in {"supported", "contradicted"} and not evidence:
            status = "no_evidence_found"
        elif status in {"supported", "contradicted"} and not quote_found:
            status, evidence = "evidence_not_verified", ""
        if status in {"not_found", "unclear", "no_evidence_found"}:
            evidence, quote_found = "", False
        results.append({
            "claim": claim, "assessment": status, "source_evidence": evidence,
            "quote_found_in_model_input": bool(quote_found),
            "claim_numbers_not_in_summary": claim_numbers_not_in_summary
        })
    if not results:
        raise ValueError("No usable claims returned. Rerun or inspect the summary manually.")
    return results, tokens, seconds, raw

def numeric_tokens(text):
    """Return normalized digit sequences so changed dates and figures can be flagged."""
    normalized = unicodedata.normalize("NFKC", str(text))
    return re.findall(r"(?<![\w])\d+(?:[.,]\d+)*(?![\w])", normalized)

def unsupported_numeric_tokens(source_text, summary):
    source_values = set(numeric_tokens(source_text))
    return sorted(set(value for value in numeric_tokens(summary) if value not in source_values))

_MONTH_ALIASES = {
    "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
    "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7,
    "july": 7, "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
}
_DATE_PATTERN = re.compile(
    r"\b(?:(?P<day>\d{1,2})\s+(?P<month>January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+(?P<year>\d{2,4})|"
    r"(?P<month_first>January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+(?P<day_first>\d{1,2})(?:st|nd|rd|th)?[,]?\s+(?P<year_first>\d{2,4}))\b",
    flags=re.I,
)

def _date_parts(match):
    day = int(match.group("day") or match.group("day_first"))
    month_text = match.group("month") or match.group("month_first")
    year = int(match.group("year") or match.group("year_first"))
    return day, _MONTH_ALIASES[month_text.casefold()], year

def _identifier_number(suffix):
    if not suffix:
        return None
    if suffix.isdigit():
        return int(suffix)
    values = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    roman = suffix.upper()
    total, previous = 0, 0
    for letter in reversed(roman):
        value = values.get(letter)
        if value is None:
            return None
        total += -value if value < previous else value
        previous = max(previous, value)
    return total

def _entity_parts(token):
    match = re.fullmatch(r"(?P<base>[A-Za-z][A-Za-z0-9]*?)(?:[-–](?P<suffix>[0-9]+|[IVXLCDM]+))?", token)
    if not match:
        return token.casefold(), None
    return match.group("base").casefold(), _identifier_number(match.group("suffix"))

def correct_source_name_spellings(source_text, summary):
    """Correct only unique, high-confidence source spellings of capitalized names."""
    token_pattern = re.compile(r"\b[A-Z][A-Za-z0-9]*(?:[-–](?:[0-9]+|[IVXLCDM]+))?\b")
    source_candidates = []
    for match in token_pattern.finditer(source_text):
        prefix = source_text[:match.start()]
        boundary = max(prefix.rfind(mark) for mark in (".", "!", "?", "।"))
        numbered_identifier = bool(re.search(r"[-–](?:[0-9]+|[IVXLCDM]+)$", match.group(0), re.I))
        if prefix[boundary + 1:].strip() or numbered_identifier:
            source_candidates.append(match.group(0))
    corrections = []
    corrected_summary = summary
    seen = set()
    for match in list(token_pattern.finditer(summary)):
        generated = match.group(0)
        if generated.casefold() in seen:
            continue
        seen.add(generated.casefold())
        generated_base, generated_number = _entity_parts(generated)
        if len(generated_base) < 6:
            continue
        candidates = []
        for source_name in source_candidates:
            source_base, source_number = _entity_parts(source_name)
            if source_number != generated_number:
                continue
            score = difflib.SequenceMatcher(None, generated_base, source_base).ratio()
            if score >= 0.90:
                candidates.append((score, source_name))
        candidates.sort(reverse=True)
        if not candidates or (len(candidates) > 1 and candidates[0][0] - candidates[1][0] < 0.06):
            continue
        best_score, source_name = candidates[0]
        if generated.casefold() == source_name.casefold():
            continue
        pattern = re.compile(rf"(?<![A-Za-z0-9]){re.escape(generated)}(?![A-Za-z0-9])", re.I)
        corrected_summary = pattern.sub(lambda _: source_name, corrected_summary)
        corrections.append({"generated_name": generated, "source_name": source_name, "similarity": round(best_score, 3)})
    return corrected_summary, corrections

def correct_unique_source_date(source_text, summary):
    """Replace one clearly mismatched full date only when each text has exactly one."""
    source_dates = list(_DATE_PATTERN.finditer(source_text))
    summary_dates = list(_DATE_PATTERN.finditer(summary))
    if len(source_dates) != 1 or len(summary_dates) != 1:
        return summary, None
    source_match, summary_match = source_dates[0], summary_dates[0]
    if _date_parts(source_match) == _date_parts(summary_match):
        return summary, None
    source_spelling = source_match.group(0)
    corrected = summary[:summary_match.start()] + source_spelling + summary[summary_match.end():]
    return corrected, {"generated_date": summary_match.group(0), "source_date": source_spelling}

def unsupported_full_dates(source_text, summary):
    """Return any complete summary dates that do not occur in the source."""
    source_dates = {_date_parts(match) for match in _DATE_PATTERN.finditer(source_text)}
    return sorted({
        match.group(0)
        for match in _DATE_PATTERN.finditer(summary)
        if _date_parts(match) not in source_dates
    })

def summary_requires_review(claims, claim_check_error=None, number_mismatches=None, date_mismatches=None, date_correction=None, name_corrections=None):
    if claim_check_error or number_mismatches or date_mismatches or date_correction or name_corrections or not claims:
        return True
    return any(
        claim.get("assessment") != "supported"
        or not claim.get("quote_found_in_model_input")
        or claim.get("independent_nli_label") == "contradiction"
        for claim in claims
    )

def factuality_summary(claims):
    """Describe evidence outcomes without presenting model judgments as probabilities."""
    if not claims:
        return "No factual claims detected — no factuality conclusion can be made."
    statuses = {claim["assessment"] for claim in claims}
    if "checker_mismatch" in statuses:
        return "Claim checker added number/date details absent from the summary — its claim results are unreliable."
    if "evidence_not_verified" in statuses:
        return "Evidence not verified — at least one evidence quote was not found verbatim in the source."
    if "no_evidence_found" in statuses:
        return "No factual evidence detected for at least one claim."
    if "contradicted" in statuses:
        return "Potential contradiction detected — inspect the cited source text."
    if "not_found" in statuses:
        return "No supporting evidence located for at least one claim."
    if "unclear" in statuses:
        return "Evidence is unclear for at least one claim — manual review is needed."
    return "A source quote was found for every claim; the AI support judgments still need review."

## 3. Paste a source and run the evaluation

Paste a complete Roman-script Hinglish passage. Inputs longer than 2,048 model tokens are truncated at the end and clearly flagged. Claim checking uses the same model-visible text as summarization.

In [ ]:
source_box = widgets.Textarea(
    value="",
    placeholder="Paste Devanagari Hindi, Devanagari+English code-mixed text, or Roman-script Hinglish here.",
    description="Source text",
    layout=widgets.Layout(width="100%", height="220px"),
    style={"description_width": "initial"}
)
model_picker = widgets.Dropdown(
    options=list(MODEL_IDS.keys()), value="Qwen3 4B",
    description="Model", style={"description_width": "initial"}
)
prompt_picker = widgets.Dropdown(
    options=["Factuality-aware", "Standard"], value="Factuality-aware",
    description="Prompt", style={"description_width": "initial"}
)
script_picker = widgets.Dropdown(
    options=["Auto-detect", "Devanagari / Code-Mixed (Hindi + English)", "Roman-script Hinglish"],
    value="Auto-detect",
    description="Archetype", style={"description_width": "initial"}
)
run_button = widgets.Button(
    description="Summarize and check claims", button_style="success", icon="play",
    layout=widgets.Layout(width="260px")
)
download_button = widgets.Button(
    description="Download last result (JSON)", icon="download", disabled=True
)
_current_progress = 0

def render_circular_progress(percent, state="info"):
    pct = max(0, min(100, int(percent)))
    colors = {"info": "#2563eb", "success": "#0d9488", "warning": "#d97706", "error": "#dc2626"}
    ring_color = colors.get(state, colors["info"])
    radius = 32
    circumference = 2 * 3.14159265 * radius
    offset = circumference * (1 - pct / 100.0)
    return (
        f'<div style="display:flex;align-items:center;justify-content:center;width:84px;height:84px;">'
        f'<svg width="84" height="84" viewBox="0 0 84 84">'
        f'<circle cx="42" cy="42" r="{radius}" fill="none" stroke="#e2e8f0" stroke-width="7"/>'
        f'<circle cx="42" cy="42" r="{radius}" fill="none" stroke="{ring_color}" stroke-width="7" '
        f'stroke-linecap="round" stroke-dasharray="{circumference:.2f}" stroke-dashoffset="{offset:.2f}" '
        f'transform="rotate(-90 42 42)"/>'
        f'<text x="42" y="47" text-anchor="middle" font-family="system-ui, -apple-system, sans-serif" '
        f'font-size="16" font-weight="700" fill="{ring_color}">{pct}%</text>'
        f'</svg></div>'
    )

progress_ring = widgets.HTML(
    value=render_circular_progress(0, "info"),
    layout=widgets.Layout(width="92px", height="88px")
)
status_display = widgets.HTML(
    value='<div style="padding:12px 14px;border-left:4px solid #2563eb;background:#f8fafc;border-radius:8px;color:#0f172a"><b>Ready.</b> Paste Roman-script Hinglish and click the button to start.</div>',
    layout=widgets.Layout(flex="1 1 auto")
)
result_output = widgets.Output()

def set_run_status(message, percent, state="info"):
    global _current_progress
    pct = max(0, min(100, int(percent)))
    _current_progress = pct
    colors = {"info": "#2563eb", "success": "#0d9488", "warning": "#d97706", "error": "#dc2626"}
    bg_colors = {"info": "#eff6ff", "success": "#f0fdfa", "warning": "#fffbeb", "error": "#fef2f2"}
    color = colors.get(state, colors["info"])
    bg = bg_colors.get(state, "#f8fafc")
    progress_ring.value = render_circular_progress(pct, state)
    status_display.value = (
        f'<div style="padding:12px 14px;border-left:4px solid {color};background:{bg};border-radius:8px;color:#0f172a">'
        f'<div style="font-size:11px;font-weight:700;letter-spacing:0.06em;color:{color};text-transform:uppercase;margin-bottom:2px">{state} · {pct}%</div>'
        f'<div style="font-size:13px;font-weight:600">{html.escape(str(message))}</div></div>'
    )

def on_run_clicked(_):
    global _last_result
    if run_button.disabled:
        return
    run_button.disabled = True
    download_button.disabled = True
    _last_result = None
    set_run_status("Starting… this tracker continues after the Colab cell’s green tick.", 1)
    with result_output:
        clear_output(wait=False)
        source = source_box.value.strip()
        if not source:
            display(Markdown("**Paste source text first.**"))
            set_run_status("Waiting for source text.", 0, "warning")
            run_button.disabled = False
            return
        try:
            archetype = detect_input_archetype(source, script_picker.value)
        except ValueError as val_err:
            display(Markdown(f"**Input check:** {val_err}"))
            set_run_status(str(val_err), 0, "warning")
            run_button.disabled = False
            return
        archetype_label = (
            "Devanagari / Code-Mixed" if archetype == "devanagari_codemixed" else "Roman-script Hinglish"
        )
        try:
            selected_model = model_picker.value
            selected_prompt = prompt_picker.value
            set_run_status("Loading model (first use downloads model weights)…", 10)
            model, tokenizer = load_model(selected_model)
            source_used, original_tokens, source_truncated = prepare_source(source, tokenizer)
            fact_anchors = extract_fact_anchors(source_used, archetype=archetype)
            set_run_status("Generating English summary…", 35)
            summary, summary_tokens, summary_seconds, summary_cap_hit = make_summary(
                source_used, selected_prompt, model, tokenizer, fact_anchors=fact_anchors, archetype=archetype
            )
            if not summary:
                raise ValueError("The model returned an empty summary. Try the other model size.")
            _, initial_date_correction = correct_unique_source_date(source_used, summary)
            initial_name_candidate, initial_name_corrections = correct_source_name_spellings(source_used, summary)
            summary_numeric_mismatches = unsupported_numeric_tokens(source_used, summary)
            summary_date_mismatches = unsupported_full_dates(source_used, summary)
            summary_numeric_retry_used = False
            summary_numeric_retry_error = None
            if summary_numeric_mismatches or summary_date_mismatches or initial_date_correction or initial_name_corrections:
                summary_numeric_retry_used = True
                set_run_status("Number/date mismatch found; retrying the summary once…", 50, "warning")
                try:
                    retry_summary, retry_tokens, retry_seconds, retry_cap_hit = make_summary(
                        source_used, selected_prompt, model, tokenizer, numeric_retry=True, fact_anchors=fact_anchors, archetype=archetype
                    )
                    if retry_summary:
                        summary = retry_summary
                        summary_tokens += retry_tokens
                        summary_seconds += retry_seconds
                        summary_cap_hit = retry_cap_hit
                except Exception as retry_error:
                    # Keep the original draft visible; the final guards below still apply.
                    summary_numeric_retry_error = f"{type(retry_error).__name__}: {retry_error}"
            summary, summary_name_corrections = correct_source_name_spellings(source_used, summary)
            summary, summary_date_correction = correct_unique_source_date(source_used, summary)
            summary_numeric_mismatches = unsupported_numeric_tokens(source_used, summary)
            summary_date_mismatches = unsupported_full_dates(source_used, summary)
            set_run_status("Checking summary claims against the source…", 65)
            claim_check_error = None
            try:
                claims, audit_tokens, audit_seconds, raw_audit = check_claims(
                    source_used, summary, model, tokenizer
                )
            except Exception as error:
                # Preserve a usable summary, but never imply it passed a check.
                claims = []
                audit_tokens, audit_seconds = 0, 0.0
                raw_audit = None
                claim_check_error = f"{type(error).__name__}: {error}"
            set_run_status("Running independent cross-lingual NLI check…", 80)
            nli_error = None
            if claims:
                for claim_item in claims:
                    if claim_item["assessment"] == "checker_mismatch":
                        claim_item["independent_nli_label"] = "skipped"
                        claim_item["independent_nli_score"] = None
                        continue
                    try:
                        premise = claim_item["source_evidence"] if claim_item["quote_found_in_model_input"] else source_used
                        nli_label, nli_score = nli_check_claim(premise, claim_item["claim"], archetype=archetype)
                        claim_item["independent_nli_label"] = nli_label
                        claim_item["independent_nli_score"] = nli_score
                    except Exception as nli_exc:
                        nli_error = f"{type(nli_exc).__name__}: {nli_exc}"
                        claim_item["independent_nli_label"] = "unavailable"
                        claim_item["independent_nli_score"] = None
            if summary_numeric_mismatches:
                factuality_status = (
                    "Possible number/date mismatch — the summary contains numeric values "
                    "not present in the source; review before use."
                )
            elif claim_check_error:
                factuality_status = "Claim check unavailable — no factuality conclusion can be made."
            else:
                factuality_status = factuality_summary(claims)
                if any(c.get("independent_nli_label") == "contradiction" for c in claims):
                    factuality_status += " Independent NLI flagged a possible contradiction — inspect carefully."
            summary_needs_review = summary_requires_review(
                claims, claim_check_error, summary_numeric_mismatches,
                summary_date_mismatches, summary_date_correction, summary_name_corrections
            )
            set_run_status("Preparing the result…", 90)
            _last_result = {
                "project": "FactIndic interactive research demo",
                "model": MODEL_IDS[selected_model],
                "independent_nli_model": NLI_MODEL_ID,
                "independent_nli_note": "mDeBERTa-v3-base-mnli-xnli is trained on Devanagari Hindi/English XNLI, not Roman-script Hinglish; treat as an unvalidated secondary signal.",
                "independent_nli_error": nli_error,
                "prompt": selected_prompt,
                "input_archetype": archetype,
                "model_visible_source_sha256": hashlib.sha256(source_used.encode("utf-8")).hexdigest(),
                "source_tokens_before_cap": original_tokens,
                "source_truncated": source_truncated,
                "summary": summary,
                "summary_requires_manual_review": summary_needs_review,
                "summary_review_status": "unverified_draft" if summary_needs_review else "AI_supported_not_human_verified",
                "summary_numeric_values_not_in_source": summary_numeric_mismatches,
                "summary_dates_not_in_source": summary_date_mismatches,
                "summary_numeric_retry_used": summary_numeric_retry_used,
                "summary_numeric_retry_error": summary_numeric_retry_error,
                "summary_date_correction": summary_date_correction,
                "summary_name_corrections": summary_name_corrections,
                "summary_tokens": summary_tokens,
                "summary_generation_seconds": round(summary_seconds, 3),
                "summary_token_cap_hit": summary_cap_hit,
                "claim_check_tokens": audit_tokens,
                "claim_check_seconds": round(audit_seconds, 3),
                "claims": claims,
                "claim_check_completed": claim_check_error is None,
                "claim_check_error": claim_check_error,
                "factuality_status": factuality_status,
                "quote_validation": "displayed evidence quotes checked against the model-visible source",
                "all_displayed_quotes_found": all((not c["source_evidence"]) or c["quote_found_in_model_input"] for c in claims)
            }
            badge_bg = "#fef3c7" if summary_needs_review else "#ccfbf1"
            badge_fg = "#92400e" if summary_needs_review else "#0f766e"
            badge_text = "UNVERIFIED DRAFT · REVIEW REQUIRED" if summary_needs_review else "AI-SUPPORTED DRAFT · NOT HUMAN-VERIFIED"
            supported_cnt = sum(1 for c in claims if c.get("assessment") == "supported")
            quotes_cnt = sum(1 for c in claims if c.get("quote_found_in_model_input"))
            contradiction_cnt = sum(1 for c in claims if c.get("independent_nli_label") == "contradiction")
            notes_html = []
            if source_truncated:
                notes_html.append(f"<li><b>Input truncated:</b> {original_tokens} tokens provided; model saw first {MAX_SOURCE_TOKENS}.</li>")
            if summary_cap_hit:
                notes_html.append("<li><b>Token cap reached:</b> inspect the summary for a truncated ending.</li>")
            if summary_name_corrections:
                corr_str = "; ".join(f"<code>{html.escape(i['generated_name'])}</code> &rarr; <code>{html.escape(i['source_name'])}</code>" for i in summary_name_corrections)
                notes_html.append(f"<li><b>Name spelling correction:</b> {corr_str}</li>")
            if summary_date_correction:
                notes_html.append(f"<li><b>Date correction:</b> <code>{html.escape(summary_date_correction['generated_date'])}</code> &rarr; <code>{html.escape(summary_date_correction['source_date'])}</code></li>")
            if summary_numeric_mismatches or summary_date_mismatches:
                mvals = ", ".join(html.escape(v) for v in (summary_numeric_mismatches + summary_date_mismatches))
                notes_html.append(f"<li><b>Number/date mismatch:</b> {mvals}</li>")
            notes_block = f'<ul style="margin:10px 0 0 18px;padding:0;color:#92400e;font-size:12.5px">{"".join(notes_html)}</ul>' if notes_html else ""

            anchor_items = []
            if fact_anchors.get("dates"):
                anchor_items.append(f"<b>Dates:</b> {html.escape(', '.join(fact_anchors['dates']))}")
            if fact_anchors.get("numbers"):
                anchor_items.append(f"<b>Numbers:</b> {html.escape(', '.join(fact_anchors['numbers']))}")
            if fact_anchors.get("cues"):
                anchor_items.append(f"<b>Polarity/warning cues:</b> {html.escape(', '.join(fact_anchors['cues']))}")
            anchors_html = " &nbsp;|&nbsp; ".join(anchor_items) if anchor_items else "No numeric, date, or warning cues extracted"

            rows_html = []
            for idx, c in enumerate(claims, 1):
                assess = c.get("assessment", "unclear")
                a_bg = "#dcfce7" if assess == "supported" else "#fee2e2"
                a_fg = "#166534" if assess == "supported" else "#991b1b"
                q_ok = bool(c.get("quote_found_in_model_input"))
                q_badge = '<span style="color:#0f766e;font-weight:700">Verified</span>' if q_ok else '<span style="color:#b45309;font-weight:700">Unverified</span>'
                ev_text = html.escape(c.get("source_evidence") or "—")
                nli_lbl = c.get("independent_nli_label") or "—"
                nli_sc = c.get("independent_nli_score")
                nli_str = f"{html.escape(nli_lbl)} ({nli_sc:.3f})" if isinstance(nli_sc, (int, float)) else html.escape(str(nli_lbl))
                nli_color = "#dc2626" if nli_lbl == "contradiction" else ("#0f766e" if nli_lbl == "entailment" else "#475569")
                rows_html.append(
                    f'<tr style="border-top:1px solid #e2e8f0;vertical-align:top">'
                    f'<td style="padding:10px 8px;color:#64748b;font-weight:600">{idx}</td>'
                    f'<td style="padding:10px 8px;color:#0f172a">{html.escape(c.get("claim", ""))}</td>'
                    f'<td style="padding:10px 8px"><span style="background:{a_bg};color:{a_fg};padding:3px 8px;border-radius:999px;font-size:11px;font-weight:700;text-transform:uppercase">{html.escape(assess)}</span></td>'
                    f'<td style="padding:10px 8px;color:#334155"><div>{q_badge}</div><div style="font-size:12px;margin-top:3px;color:#475569">"{ev_text}"</div></td>'
                    f'<td style="padding:10px 8px;font-family:monospace;font-size:12px;color:{nli_color};font-weight:600">{nli_str}</td>'
                    f'</tr>'
                )

            if claims:
                claims_section = (
                    '<table style="width:100%;border-collapse:collapse;font-size:13px;margin-top:8px">'
                    '<thead><tr style="background:#f8fafc;color:#475569;text-align:left;font-size:11px;text-transform:uppercase;letter-spacing:0.05em">'
                    '<th style="padding:9px 8px">#</th><th style="padding:9px 8px">Claim</th>'
                    '<th style="padding:9px 8px">Status</th><th style="padding:9px 8px">Source Quote</th>'
                    '<th style="padding:9px 8px">mDeBERTa NLI</th></tr></thead>'
                    f'<tbody>{"".join(rows_html)}</tbody></table>'
                )
            elif claim_check_error:
                claims_section = f'<div style="padding:12px;background:#fef2f2;border-radius:8px;color:#991b1b;font-size:13px"><b>Claim check unavailable:</b> {html.escape(claim_check_error)}</div>'
            else:
                claims_section = '<div style="padding:12px;background:#f8fafc;border-radius:8px;color:#475569;font-size:13px">No factual claims returned.</div>'

            card_html = (
                '<div style="font-family:system-ui,-apple-system,sans-serif;border:1px solid #cbd5e1;border-radius:12px;overflow:hidden;background:#ffffff;color:#0f172a;margin-top:8px">'
                '<div style="padding:14px 18px;background:#0f172a;color:#f8fafc;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px">'
                f'<div><div style="font-size:11px;letter-spacing:0.08em;color:#2dd4bf;font-weight:700;text-transform:uppercase">FactIndic Evaluation Report</div>'
                f'<div style="font-size:13px;color:#cbd5e1;margin-top:2px">{html.escape(selected_model)} &middot; {html.escape(selected_prompt)} prompt &middot; {html.escape(archetype_label)}</div></div>'
                f'<span style="background:{badge_bg};color:{badge_fg};padding:5px 11px;border-radius:999px;font-size:11px;font-weight:700">{badge_text}</span>'
                '</div>'
                '<div style="padding:18px">'
                '<div style="font-size:11px;font-weight:700;letter-spacing:0.06em;color:#64748b;text-transform:uppercase;margin-bottom:6px">Generated English Summary</div>'
                f'<div style="padding:14px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;font-size:14.5px;line-height:1.55;color:#0f172a">{html.escape(summary)}</div>'
                f'{notes_block}'
                '<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin:16px 0">'
                f'<div style="padding:10px 12px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px"><div style="font-size:11px;color:#64748b">Claims Supported</div><div style="font-size:18px;font-weight:700;color:#0f766e">{supported_cnt} / {len(claims)}</div></div>'
                f'<div style="padding:10px 12px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px"><div style="font-size:11px;color:#64748b">Quotes Verified</div><div style="font-size:18px;font-weight:700;color:#0f766e">{quotes_cnt} / {len(claims)}</div></div>'
                f'<div style="padding:10px 12px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px"><div style="font-size:11px;color:#64748b">NLI Contradictions</div><div style="font-size:18px;font-weight:700;color:{("#dc2626" if contradiction_cnt else "#0f766e")}">{contradiction_cnt}</div></div>'
                f'<div style="padding:10px 12px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px"><div style="font-size:11px;color:#64748b">Gen / Check Time</div><div style="font-size:18px;font-weight:700;color:#2563eb">{summary_seconds:.2f}s / {audit_seconds:.2f}s</div></div>'
                '</div>'
                f'<div style="padding:10px 12px;background:#f0fdfa;border:1px solid #99f6e4;border-radius:8px;font-size:12.5px;color:#115e59;margin-bottom:14px"><b>Factuality Status:</b> {html.escape(factuality_status)}<br><span style="color:#334155"><b>Extracted Anchors:</b> {anchors_html}</span></div>'
                '<div style="font-size:11px;font-weight:700;letter-spacing:0.06em;color:#64748b;text-transform:uppercase;margin-bottom:4px">Claim-by-Claim Verification</div>'
                f'{claims_section}'
                '<div style="margin-top:12px;font-size:11.5px;color:#64748b">Quote presence is checked verbatim against the model-visible source. Secondary NLI uses <code>mDeBERTa-v3-base-mnli-xnli</code> with Roman-script Hinglish gloss normalization.</div>'
                '</div></div>'
            )
            display(HTML(card_html))
            if claim_check_error:
                set_run_status(
                    f"Run finished; claim check unavailable: {claim_check_error}", 100, "warning"
                )
            elif summary_numeric_mismatches or summary_date_mismatches:
                set_run_status("Run finished with a number/date mismatch. Do not use this draft yet.", 100, "warning")
            elif summary_date_correction or summary_name_corrections:
                set_run_status("Run finished after a source correction; factuality still needs review.", 100, "warning")
            elif not claims or any(c["assessment"] != "supported" or c.get("independent_nli_label") == "contradiction" for c in claims):
                set_run_status("Run finished; manual factuality review is required.", 100, "warning")
            else:
                set_run_status("AI evidence check finished. Review the quotes and judgments.", 100, "success")
            download_button.disabled = False
        except Exception as error:
            _last_result = None
            set_run_status(f"Run failed: {type(error).__name__}: {error}", _current_progress, "error")
            display(Markdown(f"**Could not complete the check:** `{type(error).__name__}: {error}`. Try a shorter source or the 1.7B model."))
        finally:
            run_button.disabled = False

def on_download_clicked(_):
    if not _last_result:
        return
    output_path = Path("/content/factindic_interactive_result.json")
    output_path.write_text(json.dumps(_last_result, ensure_ascii=False, indent=2), encoding="utf-8")
    files.download(str(output_path))

run_button._click_handlers.callbacks.clear()
download_button._click_handlers.callbacks.clear()
run_button.on_click(on_run_clicked)
download_button.on_click(on_download_clicked)
display(widgets.VBox([
    source_box,
    widgets.HBox([model_picker, prompt_picker, script_picker]),
    widgets.HBox([run_button, download_button]),
    widgets.HTML("<b>Run progress</b>"),
    widgets.HBox([progress_ring, status_display], layout=widgets.Layout(align_items="center", gap="12px")),
    result_output
]))

## Reading the output

- **A source quote was found for every claim** means the AI reviewer supplied text that appears verbatim in the source. It does not prove the claim is correct.
- **No factual evidence detected** means the reviewer did not provide a checkable source quote for at least one claim. Treat that claim as unsupported by this run.
- **Evidence not verified** means a quote was supplied but could not be found in the source exactly; the claim is not verified.
- **Potential contradiction**, **No supporting evidence located**, or **Unclear** needs human inspection.
- If no factual claims are detected, the notebook says that no factuality conclusion can be made. It does not invent a probability score.

The progress tracker follows the button-triggered model work. The green tick beside a Colab code cell only indicates that the cell which created the controls has finished. The same model performs both summarization and checking, so use this as a live project demonstration and AI-assisted review aid.
